# 06｜三角、指數、對數的微分：週期、成長與衰減

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/06_trig_exp_log.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus06 三角與超越函數微分**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def d(f, a, h=1e-6):
    "中央差分：用前後兩點估計導數（老師提供）。"
    return (f(a + h) - f(a - h)) / (2 * h)

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. sin x 的導數是 cos x

用電腦算 $\sin x$ 每一點的斜率，再和 $\cos x$ 疊在一起。

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 400)

plt.figure(figsize=(8, 3.5))
plt.plot(x, np.sin(x), color="#0B2E33", lw=2, label="sin x")
plt.plot(x, d(np.sin, x), color="#028090", lw=6, alpha=0.4, label="numeric slope of sin x")
plt.plot(x, np.cos(x), color="#D9534F", lw=1.5, ls="--", label="cos x")
plt.xlabel("x (radians)"); plt.grid(alpha=0.3); plt.legend(); plt.show()

兩條線完全重疊。

**為什麼一定要用弧度？** 若 x 用「度」，$(\sin x)'$ 會多一個 $\pi/180$。下面驗證。

In [ ]:
sin_deg = lambda deg: np.sin(np.radians(deg))
print("sin(度) 在 0° 的斜率：", round(d(sin_deg, 0), 5))
print("π/180              ：", round(np.pi / 180, 5))
print("sin(弧度) 在 0 的斜率：", round(d(np.sin, 0), 5))

## 2. ✏️ 動手 1：cos x 的導數

$(\cos x)'=-\sin x$。請手算 $\cos x$ 在 $x=\pi/2$ 的導數，再由電腦驗證。

In [ ]:
answer_1 = None      # ✏️ 填空：cos x 在 x = π/2 的導數是多少？

# ---- 自動檢查（不用改）----
if answer_1 is None:
    print("還沒填，算出 −sin(π/2)。")
else:
    check(abs(answer_1 - d(np.cos, np.pi / 2)) < 1e-3, "sin(π/2) = 1，別忘了負號。")

## 3. 血壓：哪一刻上升最快？

$P(t)=100+20\sin(2.5\pi t)$（mmHg，t 為秒，週期 0.8 秒，示意數值）。用連鎖律：

$$P'(t)=20\cdot2.5\pi\cos(2.5\pi t)=50\pi\cos(2.5\pi t)$$

**導數 = 0 的時刻是最高點或最低點；|導數| 最大的時刻是變化最快的時刻。**

In [ ]:
P = lambda t: 100 + 20 * np.sin(2.5 * np.pi * t)
t = np.linspace(0, 1.6, 400)

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(t, P(t), color="#028090", lw=3); ax[0].set_ylabel("P (mmHg)"); ax[0].grid(alpha=0.3)
ax[1].plot(t, d(P, t), color="#D9534F", lw=3); ax[1].axhline(0, color="gray", lw=1)
ax[1].set_ylabel("P' (mmHg/s)"); ax[1].set_xlabel("t (seconds)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

for s in [0, 0.2, 0.4]:
    print(f"t = {s}：P = {P(s):.1f} mmHg，P′ ≈ {d(P, s):.1f} mmHg/s")

## 4. ✏️ 動手 2：虛擬資料中的晝夜節律

體溫模型 $T(t)=36.8+0.4\sin\!\big(\pi(t-10)/12\big)$。下面產生 7 天、每小時一筆的**虛擬**量測，再把同一小時平均起來。雜訊被平均掉之後，節律就浮現了。數據的最高點可能和模型差一小時左右，這就是雜訊的影響。

In [ ]:
rng = np.random.default_rng(24)
hours = np.arange(24 * 7)
T = lambda t: 36.8 + 0.4 * np.sin(np.pi * (t - 10) / 12)
temp = T(hours) + rng.normal(0, 0.15, hours.size)
mean_by_hour = np.array([temp[hours % 24 == h].mean() for h in range(24)])

plt.figure(figsize=(8, 3.5))
plt.plot(hours % 24, temp, ".", color="#BFDEDE", label="7 days of virtual data")
plt.plot(range(24), mean_by_hour, "o-", color="#028090", label="mean by hour")
plt.xlabel("hour of day"); plt.ylabel("T (°C)"); plt.grid(alpha=0.3); plt.legend(); plt.show()

print("數據中平均最高的時刻：", int(mean_by_hour.argmax()), "點")
print("模型 T′(16) ≈", round(d(T, 16), 4), "   T(16) =", round(T(16), 1))
print("模型 T′(10) ≈", round(d(T, 10), 3), "°C/小時（一天中升得最快）")

answer_2 = None      # ✏️ 填空：T′ = 0 且體溫最高的時刻是幾點？

# ---- 自動檢查（不用改）----
if answer_2 is None:
    print("還沒填，看上面印出的 T′ 哪一個幾乎是 0。")
else:
    check(answer_2 == 16, "T′(16) 幾乎是 0，而且 16 點的體溫最高。")

## 5. eˣ：導數就是自己

$(e^x)'=e^x$：函數值多大，斜率就多大。其他底數則多一個 $\ln a$：$(a^x)'=a^x\ln a$。

In [ ]:
print("  x     eˣ       eˣ 的斜率")
for v in [-1, 0, 1, 2]:
    print(f"{v:>3}   {np.exp(v):7.4f}   {d(np.exp, v):7.4f}")

print()
print("2ˣ 在 x = 1 的斜率：", round(d(lambda x: 2**x, 1), 4))
print("公式 2¹ · ln 2     ：", round(2 * np.log(2), 4))

## 6. 藥物衰減：從虛擬數據找半衰期

$C(t)=80e^{-0.2t}$（mg/L，t 為小時）。連鎖律給出

$$C'(t)=-0.2\cdot80e^{-0.2t}=-0.2\,C(t)$$

**衰減速度和現有濃度成正比。** 下面用**虛擬**量測值反推 k：取對數後是一條直線，斜率就是 $-k$。

In [ ]:
C = lambda t: 80 * np.exp(-0.2 * t)
print("C′(0) ≈", round(d(C, 0), 2), "   C′(5) ≈", round(d(C, 5), 2), "mg/L/h")

rng = np.random.default_rng(3)
t_obs = np.arange(0, 16, 1.5)
C_obs = C(t_obs) * np.exp(rng.normal(0, 0.05, t_obs.size))

slope = np.polyfit(t_obs, np.log(C_obs), 1)[0]
k_hat = -slope
print(f"從數據估出 k ≈ {k_hat:.3f}（真值 0.2）")
print(f"估出的半衰期 ≈ {np.log(2) / k_hat:.2f} 小時（真值 {np.log(2) / 0.2:.2f}）")

plt.figure(figsize=(6, 3.5))
plt.plot(t_obs, np.log(C_obs), "o", color="#D9534F", label="ln C (virtual data)")
plt.plot(t_obs, np.polyval(np.polyfit(t_obs, np.log(C_obs), 1), t_obs), color="#028090", lw=2, label="fitted line")
plt.xlabel("t (hours)"); plt.ylabel("ln C"); plt.grid(alpha=0.3); plt.legend(); plt.show()

## 7. ✏️ 動手 3：半衰期公式

半衰期 $t_{1/2}=\ln2/k$，和起始濃度無關。若某藥 $k=0.1$ /小時，半衰期是幾小時？用 `np.log(2)` 表示 ln 2。

In [ ]:
answer_3 = None      # ✏️ 填空：k = 0.1 時的半衰期

# ---- 自動檢查（不用改）----
if answer_3 is None:
    print("還沒填，把 None 換成 np.log(2) / 0.1。")
else:
    check(abs(answer_3 - 6.931) < 0.01, "半衰期 = ln 2 ÷ k。")

## 8. ln x 與 pH 的敏感度

$(\ln x)'=1/x$。$\mathrm{pH}=-\log_{10}[\mathrm{H}^+]$ 的導數是 $-\dfrac{1}{[\mathrm{H}^+]\ln10}$。在 pH 7.4，氫離子只多 1 nM，pH 會變多少？

In [ ]:
H0 = 10 ** -7.4
dH = 1e-9
pH = lambda H: -np.log10(H)

print("用導數估計 ΔpH ≈", round(-dH / (H0 * np.log(10)), 4))
print("直接計算   ΔpH =", round(pH(H0 + dH) - pH(H0), 4))

線性近似和真實值很接近。氫離子只多一點點，pH 就有感，這也是血液酸鹼值要控制得那麼精細的原因。

## 9. ✏️ 動手 4：腫瘤成長

$V(t)=50e^{0.05t}$（mm³，t 為天，示意數值）。$V'(t)=0.05\,V(t)$，所以**相對成長率** $V'/V$ 是固定的。改 `day` 看看。

In [ ]:
V = lambda t: 50 * np.exp(0.05 * t)
day = 10       # ← 改這裡

print(f"第 {day} 天：V ≈ {V(day):.1f} mm³，V′ ≈ {d(V, day):.2f} mm³/天，V′/V ≈ {d(V, day) / V(day):.3f}")

answer_4 = None      # ✏️ 填空：V′/V 是多少？它會隨時間改變嗎？（填數字）

# ---- 自動檢查（不用改）----
if answer_4 is None:
    print("還沒填，看上面印出的 V′/V。")
else:
    check(abs(answer_4 - 0.05) < 0.001, "看上面印出的 V′/V，每天成長 5%。")

## 10. 連結 AI：sigmoid

神經網路常用的 $\sigma(x)=\dfrac{1}{1+e^{-x}}$，導數是 $\sigma'(x)=\sigma(x)(1-\sigma(x))$。$|x|$ 很大時導數趨近 0，這是神經網路「學得慢」的原因之一。

In [ ]:
s = lambda x: 1 / (1 + np.exp(-x))
x = np.linspace(-8, 8, 300)

plt.figure(figsize=(7, 3.3))
plt.plot(x, s(x), color="#028090", lw=3, label="sigmoid")
plt.plot(x, s(x) * (1 - s(x)), color="#D9534F", lw=3, label="derivative")
plt.xlabel("x"); plt.grid(alpha=0.3); plt.legend(); plt.show()

print("σ′(0) ≈", round(d(s, 0), 3), "   σ′(2) ≈", round(d(s, 2), 3))

## 11. 用程式檢查 AI 的答案

> AI 說：$e^{-0.2t}$ 的導數還是 $e^{-0.2t}$，因為指數函數的導數是自己。

在 $t=1$ 算算看。

In [ ]:
f = lambda t: np.exp(-0.2 * t)
print("AI 的答案在 t = 1：", round(f(1), 4))
print("電腦算出的導數   ：", round(d(f, 1), 4))
print("正確公式 −0.2·e^(−0.2t)：", round(-0.2 * f(1), 4))

AI 漏了連鎖律的 $-0.2$，連正負號都錯了：衰減函數的導數一定是負的。

## 小結

- $(\sin x)'=\cos x$、$(\cos x)'=-\sin x$，x 要用弧度。
- $(e^{kt})'=ke^{kt}$：變化率和現有量成正比。
- $(\ln x)'=1/x$：對數讓相對變化變得重要。
- 週期、成長、衰減，都是身體裡常見的變化。

本週內容不在期中考範圍，期末考才會考。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*